# **Eksperimen UDAO-LASFNet pada Dataset LLVIP (RGB-IR Pedestrian Detection)**
Notebook ini menjalankan pipeline eksperimen **UDAO-LASFNet** (*Uncertainty-Driven Deformable Alignment & Differential Feature Attention Transformation*) dari repositori `Allezan/UDAO-LASFNet` pada dataset **LLVIP (`person`)**:
1. **UDAO-ASFF** (Layer 10, $P_3/8$): Penyelarasan spasial lintas-modalitas berbasis `deform_conv2d` dengan *learnable offset scale* $\gamma$.
2. **D-FATM** (Layer 24, Neck $P_3/8$): Peningkatan ketajaman tepi frekuensi tinggi (*Sobel + 8-neighbor Laplacian*).
3. **Auxiliary Boundary Loss (BAB)**: Supervisi batas objek berbasis *GT Bounding-Box Masking* ($M_{gt}$) yang hanya aktif saat *training* (**0 ms *inference overhead***).

### **LANGKAH 1: Inisialisasi Environment, Google Drive, & Clone Repositori `Allezan/UDAO-LASFNet`**

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/UDAO-LASFNet/LLVIP"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Folder target di Google Drive berhasil disiapkan di:", SAVE_DIR)

%cd /content
!rm -rf /content/UDAO-LASFNet
!git clone -b udao-dfatm https://github.com/Allezan/UDAO-LASFNet.git
%cd /content/UDAO-LASFNet

!pip install -q "scikit-image>=0.24.0" xlsxwriter huggingface_hub thop pycocotools ultralytics albumentations timm einops prettytable gitpython
print("Instalasi dependensi & clone repositori Allezan/UDAO-LASFNet selesai!")

### **LANGKAH 2: Salin & Ekstrak Dataset LLVIP dari Google Drive**

In [ ]:
import os
import shutil
import zipfile

source_on_drive = "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Dataset/LLVIP.zip"
target_zip_local = "/content/LLVIP.zip"
target_dir = "/content/UDAO-LASFNet/dataset/LLVIP"

if os.path.exists(source_on_drive):
    print(f"Menemukan dataset di Drive Anda: {source_on_drive}")
    print("Menyalin dataset ke lokal Colab...")
    shutil.copy(source_on_drive, target_zip_local)

    print("Mengekstrak dataset...")
    os.makedirs(target_dir, exist_ok=True)
    with zipfile.ZipFile(target_zip_local, 'r') as zip_ref:
        zip_ref.extractall(target_dir)
    print(f"Selesai! Dataset berhasil disiapkan di: {target_dir}")
else:
    print(f"File tidak ditemukan di: {source_on_drive}")

### **LANGKAH 3: Penataan Struktur Direktori LLVIP, Konversi XML ke YOLO TXT, & Patch PyTorch 2.6+**

In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET
import yaml

REPO_DIR = "/content/UDAO-LASFNet"
%cd /content/UDAO-LASFNet
nested_dir = "/content/UDAO-LASFNet/dataset/LLVIP/LLVIP"
root_path = "/content/UDAO-LASFNet/dataset/LLVIP"

if os.path.exists(nested_dir):
    for item in os.listdir(nested_dir):
        shutil.move(os.path.join(nested_dir, item), root_path)
    os.rmdir(nested_dir)

def organize_split(modality, split):
    current_dir = os.path.join(root_path, modality, split)
    if not os.path.exists(current_dir):
        return
    target_images_dir = os.path.join(current_dir, "images")
    if not os.path.exists(target_images_dir):
        os.makedirs(target_images_dir, exist_ok=True)
        for item in os.listdir(current_dir):
            item_path = os.path.join(current_dir, item)
            if os.path.isfile(item_path) and item.lower().endswith(('.jpg', '.jpeg', '.png')):
                shutil.move(item_path, target_images_dir)

for modality in ["visible", "infrared"]:
    for split in ["train", "test"]:
        organize_split(modality, split)

def convert_coor(size, box):
    dw, dh = 1.0 / size[0], 1.0 / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

def convert_xml_to_yolo(xml_dir, images_dir, labels_dir):
    os.makedirs(labels_dir, exist_ok=True)
    if not os.path.exists(images_dir):
        return
    image_files = {os.path.splitext(f)[0] for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))}
    for img_name in image_files:
        xml_path = os.path.join(xml_dir, f"{img_name}.xml")
        txt_path = os.path.join(labels_dir, f"{img_name}.txt")
        if not os.path.exists(xml_path):
            open(txt_path, 'w').close()
            continue
        tree = ET.parse(xml_path)
        root = tree.getroot()
        size = root.find('size')
        w, h = int(size.find('width').text), int(size.find('height').text)
        lines = []
        for obj in root.iter('object'):
            cls = obj.find('name').text
            if cls != 'person':
                continue
            xmlbox = obj.find('bndbox')
            b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text),
                 float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))
            bb = convert_coor((w, h), b)
            lines.append(f"0 {' '.join(f'{a:.6f}' for a in bb)}")
        with open(txt_path, 'w') as f:
            f.write('\n'.join(lines))

xml_annotations_dir = os.path.join(root_path, "Annotations")
for modality in ["visible", "infrared"]:
    for split in ["train", "test"]:
        convert_xml_to_yolo(
            xml_annotations_dir,
            os.path.join(root_path, modality, split, "images"),
            os.path.join(root_path, modality, split, "labels")
        )

yaml_content = {
    'path': '/content/UDAO-LASFNet/dataset/LLVIP',
    'train_rgb': 'visible/train/images',
    'train_ir': 'infrared/train/images',
    'val_rgb': 'visible/test/images',
    'val_ir': 'infrared/test/images',
    'test_rgb': 'visible/test/images',
    'test_ir': 'infrared/test/images',
    'nc': 1,
    'names': ['person']
}
with open('/content/UDAO-LASFNet/data/LLVIP.yaml', 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

# Patch kompatibilitas PyTorch 2.6+ (weights_only=False & ModelEMA deepcopy fix) pada train.py dan val.py
header_patch = '''import sys
import copy
import torch
import utils.general as ug
from models.common import D_FATM
ug.check_requirements = lambda *args, **kwargs: True
_orig_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_load(*args, **kwargs)
torch.load = _patched_load
def _dfatm_deepcopy(self, memo):
    self.edge_logits = None
    self.edge_pred = None
    cls = self.__class__
    res = cls.__new__(cls)
    memo[id(self)] = res
    for k, v in self.__dict__.items():
        setattr(res, k, copy.deepcopy(v, memo))
    return res
D_FATM.__deepcopy__ = _dfatm_deepcopy
'''
for fname in ["train.py", "val.py"]:
    fpath = os.path.join(REPO_DIR, fname)
    with open(fpath, "r") as f:
        code = f.read()
    if "_patched_load" not in code:
        with open(fpath, "w") as f:
            f.write(header_patch + "\n" + code)
print("Dataset LLVIP, data/LLVIP.yaml, & patch PyTorch 2.6+ siap digunakan!")

### **LANGKAH 4: Sanity Check & Profiling Arsitektur UDAO-LASFNet (< 8.0M Parameters)**

In [ ]:
%cd /content/UDAO-LASFNet
import torch
from models.yolo import Model

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = Model('models/udao_lasfnet.yaml', ch=3, nc=1).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal Parameters (Unfused): {total_params:,} ({total_params/1e6:.3f}M)")
assert total_params < 8_000_000, "Parameter melebihi batas 8.0M!"
print("Sanity Check LULUS: Model berada di bawah batas 8.0M parameter!")

### **LANGKAH 5: Mulai Proses Training UDAO-LASFNet pada LLVIP (300 Epochs)**

In [ ]:
%cd /content/UDAO-LASFNet

!python train.py \
  --weights "" \
  --cfg models/udao_lasfnet.yaml \
  --data data/LLVIP.yaml \
  --hyp data/hyps/hyp.scratch.yaml \
  --epochs 300 \
  --batch-size 48 \
  --imgsz 640 \
  --device 0 \
  --workers 8 \
  --project "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/UDAO-LASFNet/LLVIP" \
  --name "Full UDAO-LASFNet" \
  --exist-ok

### **LANGKAH 6: Evaluasi Model Terbaik (`best.pt`) pada Test Split LLVIP**

In [ ]:
%cd /content/UDAO-LASFNet

!python val.py \
  --data data/LLVIP.yaml \
  --weights "/content/drive/MyDrive/Skripsi (Tugas Akhir)/Laboratory/UDAO-LASFNet/LLVIP/Full UDAO-LASFNet/weights/best.pt" \
  --batch-size 32 \
  --imgsz 640 \
  --device 0 \
  --task test \
  --verbose